<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Regression Model Development (Capstone)

*Session 5 · Notebook 02.07 · Practical · Coach version*

## About this notebook

This is the capstone for the regression module. You will take a real dataset and run the whole regression workflow end to end: check its quality, explore it, then build and compare a series of models (simple and multiple linear regression, and the regularised Ridge and Lasso). The goal is to practise the full process a modeller follows, not just fit one model, and to finish with an evidence-based recommendation.

**scikit-learn documentation for the model(s) used in this notebook:** [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) and [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) and [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

## About the exercises

Every exercise has three cells: a **question** (markdown), an empty **`# Your turn`** cell for you to attempt it, and a **`# Coach answer`** cell with a worked solution. Try each one before revealing the answer. The coach version runs top to bottom, and later parts reuse variables created earlier (especially one shared train/test split, so the final model comparison is fair).

## About the data

The **automobile** dataset (`automobile.csv`, 201 cars) describes each car's specifications (engine size, weight, horsepower, body style, drive wheels, ...) and its **price**, which is our regression target. It is read from the repo-root `datasets/` folder (two levels up).

## Index

- [Part 0: Setup](#p0)
- [Part 1: Data quality](#p1)
- [Part 2: Exploratory data analysis](#p2)
- [Modelling setup (shared split)](#msetup)
- [Part 3: Simple linear regression](#p3)
- [Part 4: Multiple linear regression](#p4)
- [Part 5: Regularisation (Ridge and Lasso)](#p5)
- [Part 6: Capstone - compare the models](#p6)
- [Further practice](#further)

<a id="p0"></a>
# Part 0: Setup

Load the libraries and the data. The target is `price`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# auto = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/automobile.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# auto = pd.read_csv(session_datasets_http["automobile"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# auto = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/automobile.csv", header=True, inferSchema=True).toPandas()
auto = pd.read_csv('../../datasets/Session_5/automobile.csv')
print('shape:', auto.shape)
auto.head()

<a id="p1"></a>
# Part 1: Data quality

Before modelling, check the data is sound (Session 4 and Session 5 notebook 01.03): missing values, types, duplicates and obvious outliers. A model is only as good as the data underneath it.

### Exercise 1.1: Missing values and types

Print the number of missing values per column (only those with any) and the count of duplicated rows. How many values are missing in total?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
print('Missing per column:')
print(auto.isna().sum()[auto.isna().sum() > 0])
print('\nDuplicated rows:', auto.duplicated().sum())
print('Total missing:', int(auto.isna().sum().sum()))

### Exercise 1.2: Outliers in the target

Using the IQR rule, count how many cars have an outlying `price` (above Q3 + 1.5*IQR or below Q1 - 1.5*IQR), and print the upper bound.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
q1, q3 = auto['price'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = auto[(auto['price'] < low) | (auto['price'] > high)]
print(f'Price IQR upper bound: {high:,.0f}; outliers: {len(outliers)}')
print('These are genuine luxury cars, not errors, so we keep them.')

### Exercise 1.3: Handle the missing values

Only a handful of rows have missing values. Drop them (reset the index) and assign the result back to `auto`. Confirm there are no missing values left.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
auto = auto.dropna().reset_index(drop=True)
print('Rows after dropping missing:', len(auto))
print('Missing values remaining:', int(auto.isna().sum().sum()))

<a id="p2"></a>
# Part 2: Exploratory data analysis

Understand the target and its drivers before modelling: the shape of `price`, which numeric features relate to it most, and how a categorical feature relates to it.

### Exercise 2.1: Distribution of the target

Plot a histogram of `price` and print its skewness. Is it symmetric or skewed? (This hints at whether a transform might help later.)

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
print('Price skewness:', round(auto['price'].skew(), 2))
auto['price'].plot(kind='hist', bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of price'); plt.xlabel('price'); plt.show()
print('Price is right-skewed (a long tail of expensive cars).')

### Exercise 2.2: Which numeric features drive price?

Compute the correlation of every numeric feature with `price` and print the top six by absolute value. Which feature is the strongest single predictor?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
corr = auto.select_dtypes('number').corr()['price'].drop('price')
print(corr.abs().sort_values(ascending=False).head(6).round(2))
print('engine-size is the strongest single numeric predictor of price.')

### Exercise 2.3: Visualise the relationships

Draw a scatter of `engine-size` vs `price`, and a boxplot of `price` by `drive-wheels`. What do they suggest?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(data=auto, x='engine-size', y='price', ax=ax[0])
ax[0].set_title('engine-size vs price')
sns.boxplot(data=auto, x='drive-wheels', y='price', ax=ax[1])
ax[1].set_title('price by drive-wheels')
plt.tight_layout(); plt.show()
print('Price rises with engine size, and rear-wheel-drive cars are pricier on average.')

<a id="msetup"></a>
# Modelling setup (shared split)

So the model comparison at the end is fair, we fix **one** feature set, target and train/test split here and reuse them for every model. We use a compact set of strong numeric predictors plus three categorical features, and a `results` dictionary to collect each model's scores. This cell is given, run it as-is.

In [ ]:
numeric_features = ['engine-size', 'curb-weight', 'horsepower', 'width', 'highway-mpg']
categorical_features = ['drive-wheels', 'body-style', 'aspiration']

X = auto[numeric_features + categorical_features]
y = auto['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
def record(name, y_true, y_pred, n_features):
    results[name] = {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'n_features': n_features,
    }
    print(f'{name}: R2={results[name]["R2"]:.3f}, RMSE={results[name]["RMSE"]:,.0f}, '
          f'MAE={results[name]["MAE"]:,.0f}, features={n_features}')

print('Train:', X_train.shape, '| Test:', X_test.shape)

<a id="p3"></a>
# Part 3: Simple linear regression

Start simple: one predictor. Use `engine-size` (the strongest single driver from the EDA) to predict `price`.

### Exercise 3.1: Fit and evaluate an SLR model

Fit a `LinearRegression` using only `engine-size` (from `X_train[['engine-size']]`), predict on the test set, and record its scores with `record('SLR', y_test, preds, n_features=1)`.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
slr = LinearRegression().fit(X_train[['engine-size']], y_train)
slr_pred = slr.predict(X_test[['engine-size']])
record('SLR (engine-size)', y_test, slr_pred, n_features=1)

### Exercise 3.2: Interpret it

Print the SLR slope and intercept, and write the equation for price in terms of engine size.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
print(f'Slope: {slr.coef_[0]:,.1f} per unit of engine size')
print(f'Intercept: {slr.intercept_:,.0f}')
print(f'price = {slr.intercept_:,.0f} + {slr.coef_[0]:,.1f} * engine-size')

<a id="p4"></a>
# Part 4: Multiple linear regression

Now use all the features. Numeric features are scaled and categorical features one-hot encoded, inside a leakage-safe `ColumnTransformer` pipeline. This is the pattern for every model below.

### Exercise 4.1: Build and evaluate an MLR pipeline

Build a `ColumnTransformer` (`StandardScaler` on `numeric_features`, `OneHotEncoder` on `categorical_features`), put it in a `Pipeline` with `LinearRegression`, fit on the training data, and record its scores as `'MLR'`. (For `n_features`, use the number of model coefficients.)

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])
mlr = Pipeline([('prep', preprocess), ('clf', LinearRegression())])
mlr.fit(X_train, y_train)
n = len(mlr.named_steps['clf'].coef_)
record('MLR', y_test, mlr.predict(X_test), n_features=n)
print('Multiple features improve on the single-predictor SLR.')

<a id="p5"></a>
# Part 5: Regularisation (Ridge and Lasso)

Multiple regression can overfit when there are many features or they are correlated. Regularisation controls this. We fit **Ridge** (shrinks the coefficients) and **Lasso** (shrinks and selects features) on the model's features, choosing the penalty by cross-validation.

### Exercise 5.1: RidgeCV

Reuse the `preprocess` preprocessor from Part 4 in a pipeline with `RidgeCV(alphas=np.logspace(-2, 4, 50))`. Fit and record as `'Ridge'`.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
ridge = Pipeline([('prep', preprocess), ('clf', RidgeCV(alphas=np.logspace(-2, 4, 50)))])
ridge.fit(X_train, y_train)
n = int((ridge.named_steps['clf'].coef_ != 0).sum())
record('Ridge', y_test, ridge.predict(X_test), n_features=n)
print('Chosen alpha:', round(ridge.named_steps['clf'].alpha_, 3))

### Exercise 5.2: LassoCV

Do the same with `LassoCV(alphas=np.logspace(-2, 4, 50), cv=5, max_iter=10000)`. Record as `'Lasso'`. How many features does Lasso keep (non-zero coefficients)?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
lasso = Pipeline([('prep', preprocess),
                  ('clf', LassoCV(alphas=np.logspace(-2, 4, 50), cv=5, max_iter=10000))])
lasso.fit(X_train, y_train)
kept = int((lasso.named_steps['clf'].coef_ != 0).sum())
total = len(lasso.named_steps['clf'].coef_)
record('Lasso', y_test, lasso.predict(X_test), n_features=kept)
print(f'Lasso kept {kept} of {total} features; the rest were set to zero.')

<a id="p6"></a>
# Part 6: Capstone - compare the models

Every model was scored on the **same** test set and stored in `results`. Now bring them together and decide which to recommend, weighing accuracy against simplicity.

### Exercise 6.1: Build the comparison table

Turn the `results` dictionary into a DataFrame (models as rows; columns R2, RMSE, MAE, n_features), sorted by R2 (best first), and display it rounded.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
comparison = pd.DataFrame(results).T.sort_values('R2', ascending=False)
comparison[['R2', 'RMSE', 'MAE', 'n_features']].round({'R2': 3, 'RMSE': 0, 'MAE': 0, 'n_features': 0})

### Exercise 6.2: Conclusions and recommendation

In a few sentences, state which model you would recommend and why, referring to the table. Consider both accuracy (R2/RMSE) and simplicity (number of features). The coach answer shows one reasoned conclusion.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
best = comparison.index[0]
print('=' * 66)
print('REGRESSION MODEL COMPARISON: predicting car price')
print('=' * 66)
print(comparison[['R2', 'RMSE', 'MAE', 'n_features']].round(3).to_string())
print()
print('Conclusions:')
print(f'  - The simple one-feature model (engine-size) is a decent baseline but is beaten by '
      'the multi-feature models.')
print(f'  - The strongest test R2 is {best}.')
print('  - Regularisation (Ridge/Lasso) guards against overfitting when features are many or '
      'correlated, and Lasso also trims the feature set.')
print()
print('Recommendation: prefer the model with the best test score whose complexity is justified. '
      'If two models are close, choose the simpler / more interpretable one (fewer features), '
      'which is easier to explain and monitor, exactly the trade-off a risk team weighs.')

<a id="further"></a>
## Further practice

- `price` was right-skewed: try modelling `log(price)` and see whether the metrics improve.
- Add more categorical features (for example `make`) and see whether Lasso still trims them.
- Use `cross_val_score` instead of a single split to compare the models more robustly.
- Inspect the residuals of your best model (predicted vs actual) to check the fit.